# Linha Categórica — Múltiplos Painéis (Small Multiples)

Combina os três gráficos de linha de "Número absoluto de casos de câncer de
mama por [variável], de 1998 até 2023 em MG" (Faixa Etária, Escolaridade e
Raça/Cor) em uma única figura com três painéis (`make_subplots`), em 3
linhas x 1 coluna, compartilhando o eixo X (ano) para facilitar a comparação
visual entre os três recortes.

Os dados vêm do mesmo CSV agregado estadual por ano usado em
`area_linha_casos_categorico.ipynb` (`tabela_regional_agregada_por_ano.csv`),
recortado por `preparar_categorica_estadual_por_ano` -- nenhuma agregação é
recalculada aqui.

Cada painel mantém sua própria legenda (`legendgroup` + `legendgrouptitle`),
já que as categorias de cada variável são diferentes entre si -- uma legenda
única misturaria faixas etárias, níveis de escolaridade e raça/cor sem
distinção visual de a qual painel cada item pertence.

In [ ]:
import sys
from pathlib import Path

# Garante que o notebook encontre os módulos locais (deste notebook e de utilitarios_comuns.py)
sys.path.append(str(Path.cwd()))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()


In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utilitarios_comuns import (
    PASTA_POPULACAO,
    SUFIXO_VARIAVEL_ESTADUAL,
    NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO,
)
from utilitarios_graficos_populacionais import (
    carregar_taxa_por_regional,
    preparar_categorica_estadual_por_ano,
    remover_anos_sem_dado,
    ordenar_categorias,
    gerar_paleta_categorica_forte,
    gerar_marcadores_categoria,
    salvar_figura,
)
from configuracao_layout_graficos import (
    TAMANHO_FONTE_TITULO_GRAFICO_LINHA,
    TAMANHO_FONTE_TITULO_EIXO_GRAFICO_LINHA,
    TAMANHO_FONTE_TICK_EIXO_GRAFICO_LINHA,
    TAMANHO_FONTE_LEGENDA_GRAFICO_LINHA,
    TAMANHO_FONTE_TITULO_PAINEL_POPULACAO,
    TAMANHO_MARCADOR_LINHA_CATEGORICA,
    LARGURA_BORDA_MARCADOR_LINHA_CATEGORICA,
    COR_BORDA_MARCADOR_LINHA_CATEGORICA,
    tamanho_fonte,
)


### Constantes globais

In [3]:
PASTA_TAXAS = PASTA_POPULACAO / "Taxa"
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Linha Categórico"

SALVAR_FIGURA = True
NOME_ARQUIVO_FIGURA = "casos_multiplos_paineis_linha_categorica"

ANO_INICIAL_EXIBIDO = 1998
ANO_FINAL_EXIBIDO = 2023

NOME_COLUNA_ANO = "ano"

TITULO_GERAL_DA_FIGURA = (
    f"Número absoluto de casos de câncer de mama em MG, de {ANO_INICIAL_EXIBIDO} até "
    f"{ANO_FINAL_EXIBIDO}"
)

# Mesmo padrão de nomenclatura das colunas usado em area_linha_casos_categorico.ipynb
# (ordenar_categorias/gerar_paleta_categorica_forte já tratam "SEM INFORMAÇÃO" à parte)
NOMES_EXIBICAO_FAIXA_ETARIA = {
    "0-9": "0-9", "10-19": "10-19", "20-29": "20-29", "30-39": "30-39",
    "40-49": "40-49", "50-59": "50-59", "60-69": "60-69", "70 OU MAIS": "70 ou mais",
}
NOMES_EXIBICAO_ESCOLARIDADE = {
    "NENHUMA": "Nenhuma", "FUNDAMENTAL INCOMPLETO": "Fundamental incompleto",
    "FUNDAMENTAL COMPLETO": "Fundamental completo", "NÍVEL MÉDIO": "Nível médio",
    "NÍVEL SUPERIOR INCOMPLETO": "Nível superior incompleto",
    "NÍVEL SUPERIOR COMPLETO": "Nível superior completo", "SEM INFORMAÇÃO": "Sem informação",
}
NOMES_EXIBICAO_RACA_COR = {
    "BRANCA": "Branca", "PRETA": "Preta", "PARDA": "Parda",
    "AMARELA": "Amarela", "INDÍGENA": "Indígena", "SEM INFORMAÇÃO": "Sem informação",
}

ESPECIFICACOES_DOS_PAINEIS = [
    {
        "NOME_DO_PAINEL": "Faixa Etária",
        "TITULO_DO_PAINEL": "A) Por faixa etária",
        "NOME_DA_VARIAVEL_ESTADUAL": "Casos por Faixa Etária",
        "NOMES_EXIBICAO_POR_CATEGORIA_BRUTA": NOMES_EXIBICAO_FAIXA_ETARIA,
        "NUMERO_DA_LINHA_NO_GRID": 1,
    },
    {
        "NOME_DO_PAINEL": "Escolaridade",
        "TITULO_DO_PAINEL": "B) Por escolaridade",
        "NOME_DA_VARIAVEL_ESTADUAL": "Casos por Escolaridade",
        "NOMES_EXIBICAO_POR_CATEGORIA_BRUTA": NOMES_EXIBICAO_ESCOLARIDADE,
        "NUMERO_DA_LINHA_NO_GRID": 2,
    },
    {
        "NOME_DO_PAINEL": "Raça/Cor",
        "TITULO_DO_PAINEL": "C) Por raça/cor",
        "NOME_DA_VARIAVEL_ESTADUAL": "Casos por Raça e Cor",
        "NOMES_EXIBICAO_POR_CATEGORIA_BRUTA": NOMES_EXIBICAO_RACA_COR,
        "NUMERO_DA_LINHA_NO_GRID": 3,
    },
]

NUMERO_DE_LINHAS_NO_GRID = 3
NUMERO_DE_COLUNAS_NO_GRID = 1

LARGURA_DA_FIGURA = 1300
ALTURA_DA_FIGURA = 1500
MARGEM_DIREITA_PARA_LEGENDA = 230

ESPACAMENTO_VERTICAL_ENTRE_PAINEIS = 0.08


### Carregamento e preparação dos dados

In [4]:
tabela_agregada_estadual = carregar_taxa_por_regional(PASTA_TAXAS / NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO)

dicionario_de_dataframes_por_painel = {}
dicionario_de_paletas_por_painel = {}
dicionario_de_marcadores_por_painel = {}
dicionario_de_categorias_ordenadas_por_painel = {}

for especificacao_do_painel in ESPECIFICACOES_DOS_PAINEIS:
    nome_da_variavel_estadual = especificacao_do_painel["NOME_DA_VARIAVEL_ESTADUAL"]
    nomes_exibicao_por_categoria_bruta = especificacao_do_painel["NOMES_EXIBICAO_POR_CATEGORIA_BRUTA"]

    _tabela_pivotada_taxa, tabela_pivotada_quantidade = preparar_categorica_estadual_por_ano(
        tabela_agregada_estadual,
        SUFIXO_VARIAVEL_ESTADUAL[nome_da_variavel_estadual],
        ANO_INICIAL_EXIBIDO,
        ANO_FINAL_EXIBIDO,
    )

    # Remove os anos sem dado em nenhuma categoria e só depois preenche as ausências com 0
    (tabela_pivotada_quantidade,) = remover_anos_sem_dado(tabela_pivotada_quantidade)
    tabela_pivotada_quantidade = tabela_pivotada_quantidade.fillna(0)

    categorias_ordenadas_brutas = ordenar_categorias(tabela_pivotada_quantidade.columns.tolist())
    paleta_de_cores = gerar_paleta_categorica_forte(categorias_ordenadas_brutas)
    marcadores_de_categoria = gerar_marcadores_categoria(categorias_ordenadas_brutas)

    tabela_pivotada_quantidade = tabela_pivotada_quantidade.rename(columns=nomes_exibicao_por_categoria_bruta)
    tabela_do_painel = tabela_pivotada_quantidade.reset_index().rename(columns={"ANO": NOME_COLUNA_ANO})

    nome_do_painel = especificacao_do_painel["NOME_DO_PAINEL"]
    dicionario_de_dataframes_por_painel[nome_do_painel] = tabela_do_painel
    # Paleta/marcadores ficam indexados pelo nome de exibição, já usado nos traços do gráfico
    dicionario_de_paletas_por_painel[nome_do_painel] = {
        nomes_exibicao_por_categoria_bruta[bruta]: cor for bruta, cor in paleta_de_cores.items()
    }
    dicionario_de_marcadores_por_painel[nome_do_painel] = {
        nomes_exibicao_por_categoria_bruta[bruta]: marcador for bruta, marcador in marcadores_de_categoria.items()
    }
    dicionario_de_categorias_ordenadas_por_painel[nome_do_painel] = [
        nomes_exibicao_por_categoria_bruta[bruta] for bruta in categorias_ordenadas_brutas
    ]

df_casos_faixa_etaria = dicionario_de_dataframes_por_painel["Faixa Etária"]
df_casos_escolaridade = dicionario_de_dataframes_por_painel["Escolaridade"]
df_casos_raca_cor = dicionario_de_dataframes_por_painel["Raça/Cor"]


### Montagem da figura

In [5]:
lista_de_titulos_dos_paineis = [e["TITULO_DO_PAINEL"] for e in ESPECIFICACOES_DOS_PAINEIS]

figura_com_multiplos_paineis = make_subplots(
    rows=NUMERO_DE_LINHAS_NO_GRID,
    cols=NUMERO_DE_COLUNAS_NO_GRID,
    shared_xaxes=True,
    subplot_titles=lista_de_titulos_dos_paineis,
    vertical_spacing=ESPACAMENTO_VERTICAL_ENTRE_PAINEIS,
)

# Mesma paleta/marcador/borda/espessura de linha usados em montar_grafico_linha_categorica_absoluto
# (area_linha_casos_categorico.ipynb), para os 3 painéis ficarem visualmente consistentes com os
# gráficos individuais de cada variável.
for especificacao_do_painel in ESPECIFICACOES_DOS_PAINEIS:
    nome_do_painel = especificacao_do_painel["NOME_DO_PAINEL"]
    numero_da_linha_no_grid = especificacao_do_painel["NUMERO_DA_LINHA_NO_GRID"]
    tabela_do_painel = dicionario_de_dataframes_por_painel[nome_do_painel]
    paleta_do_painel = dicionario_de_paletas_por_painel[nome_do_painel]
    marcadores_do_painel = dicionario_de_marcadores_por_painel[nome_do_painel]
    categorias_do_painel = dicionario_de_categorias_ordenadas_por_painel[nome_do_painel]

    # Uma legenda própria por painel (legend/legend2/legend3), não uma legenda única no canto
    nome_da_legenda_do_painel = "legend" if numero_da_linha_no_grid == 1 else f"legend{numero_da_linha_no_grid}"

    for nome_da_categoria in categorias_do_painel:
        cor_da_categoria = paleta_do_painel[nome_da_categoria]
        marcador_da_categoria = marcadores_do_painel[nome_da_categoria]

        traco_da_categoria = go.Scatter(
            x=tabela_do_painel[NOME_COLUNA_ANO], y=tabela_do_painel[nome_da_categoria],
            mode="lines+markers", name=nome_da_categoria,
            legend=nome_da_legenda_do_painel,
            line=dict(color=cor_da_categoria, width=2),
            marker=dict(
                symbol=marcador_da_categoria, size=TAMANHO_MARCADOR_LINHA_CATEGORICA, color=cor_da_categoria,
                line=dict(color=COR_BORDA_MARCADOR_LINHA_CATEGORICA, width=LARGURA_BORDA_MARCADOR_LINHA_CATEGORICA),
            ),
            hovertemplate=f"{nome_da_categoria}<br>Ano: %{{x}}<br>Casos: %{{y:,.0f}}<extra></extra>",
        )
        figura_com_multiplos_paineis.add_trace(traco_da_categoria, row=numero_da_linha_no_grid, col=1)

    figura_com_multiplos_paineis.update_yaxes(
        title=dict(text="Número absoluto de casos", font=dict(size=tamanho_fonte(TAMANHO_FONTE_TITULO_EIXO_GRAFICO_LINHA))),
        tickfont=dict(size=tamanho_fonte(TAMANHO_FONTE_TICK_EIXO_GRAFICO_LINHA)),
        automargin=True,
        row=numero_da_linha_no_grid, col=1,
    )

# Números do eixo (ano) visíveis nos 3 painéis, não só no de baixo -- mesmo padrão de
# montar_paineis_populacao_total_e_por_faixa (figura.update_xaxes com showticklabels=True,
# sem restringir a `row`, já que shared_xaxes=True esconde os rótulos das linhas de cima por padrão).
# Range com folga de meia categoria de cada lado, pra não cortar os marcadores de 1998 e 2023.
FOLGA_EIXO_X_CATEGORIA = 0.4
figura_com_multiplos_paineis.update_xaxes(
    tickfont=dict(size=tamanho_fonte(TAMANHO_FONTE_TICK_EIXO_GRAFICO_LINHA)),
    type="category",
    range=[-FOLGA_EIXO_X_CATEGORIA, (ANO_FINAL_EXIBIDO - ANO_INICIAL_EXIBIDO) + FOLGA_EIXO_X_CATEGORIA],
    showticklabels=True,
)
# Título "Ano" só embaixo, pra não repetir 3 vezes
figura_com_multiplos_paineis.update_xaxes(
    title=dict(text="Ano", font=dict(size=tamanho_fonte(TAMANHO_FONTE_TITULO_EIXO_GRAFICO_LINHA))),
    automargin=True,
    row=NUMERO_DE_LINHAS_NO_GRID, col=1,
)

# Títulos A)/B)/C) em negrito, alinhados à esquerda de cada painel -- mesmo padrão dos
# painéis A/B de população (montar_paineis_populacao_total_e_por_faixa)
for anotacao in figura_com_multiplos_paineis.layout.annotations:
    anotacao.update(
        text=f"<b>{anotacao.text}</b>",
        font=dict(size=tamanho_fonte(TAMANHO_FONTE_TITULO_PAINEL_POPULACAO)),
        x=0, xanchor="left",
    )

# Uma legenda por painel, na lateral direita, centralizada verticalmente no domínio do seu próprio
# painel (em vez de uma legenda única, grudada no canto, com as 21 categorias misturadas)
configuracao_das_legendas = {}
for especificacao_do_painel in ESPECIFICACOES_DOS_PAINEIS:
    numero_da_linha_no_grid = especificacao_do_painel["NUMERO_DA_LINHA_NO_GRID"]
    nome_do_eixo_y = "yaxis" if numero_da_linha_no_grid == 1 else f"yaxis{numero_da_linha_no_grid}"
    dominio_inferior, dominio_superior = figura_com_multiplos_paineis.layout[nome_do_eixo_y].domain
    y_central_do_painel = (dominio_inferior + dominio_superior) / 2

    nome_da_legenda_do_painel = "legend" if numero_da_linha_no_grid == 1 else f"legend{numero_da_linha_no_grid}"
    configuracao_das_legendas[nome_da_legenda_do_painel] = dict(
        title=dict(text=f"<b>{especificacao_do_painel['NOME_DO_PAINEL']}</b>", font=dict(size=tamanho_fonte(TAMANHO_FONTE_LEGENDA_GRAFICO_LINHA))),
        orientation="v", yanchor="middle", y=y_central_do_painel, xanchor="left", x=1.02,
        font=dict(size=tamanho_fonte(TAMANHO_FONTE_LEGENDA_GRAFICO_LINHA - 4)),
    )

figura_com_multiplos_paineis.update_layout(
    template="plotly_white",
    width=LARGURA_DA_FIGURA, height=ALTURA_DA_FIGURA,
    margin=dict(l=20, r=MARGEM_DIREITA_PARA_LEGENDA, t=45, b=20),
    showlegend=True,
    **configuracao_das_legendas,
)

figura_com_multiplos_paineis.show()

if SALVAR_FIGURA:
    salvar_figura(figura_com_multiplos_paineis, NOME_ARQUIVO_FIGURA, PASTA_SAIDA_GRAFICOS)


[LOG] Figura PNG salva em : /home/anna/tcc/Avaliação e Apresentação/Gráficos/Linha Categórico/casos_multiplos_paineis_linha_categorica.png
[LOG] Figura SVG salva em : /home/anna/tcc/Avaliação e Apresentação/Gráficos/Linha Categórico/casos_multiplos_paineis_linha_categorica.svg
[LOG] Figura HTML salva em: /home/anna/tcc/Avaliação e Apresentação/Gráficos/Linha Categórico/casos_multiplos_paineis_linha_categorica.html
